In [1]:
''' Script permettant de spliter et constituer proprepement les jeux de données entrainement - validation - test

# Constitué après le script "0-Build_Datasets" qui a déjà permis d'équilibrer les classes et de transformer les images
# (data-augmentation)

# On part du répertoire data\processed pour arriver dans data\split

# C'est ce dernier qui sera utilisé en MLOps et déposé dans Git / DVC

'''

' Script permettant de spliter et constituer proprepement les jeux de données entrainement - validation - test\n\n# Constitué après le script "0-Build_Datasets" qui a déjà permis d\'équilibrer les classes et de transformer les images\n# (data-augmentation)\n\n# On part du répertoire data\\processed pour arriver dans data\\split\n\n# C\'est ce dernier qui sera utilisé en MLOps et déposé dans Git / DVC\n\n'

In [2]:
# import des librairies nécessaires

from pathlib import Path
import re
import shutil
import numpy as np
import csv

In [3]:
# Chemins d'accès

rep_data = "C:\\Users\\Utilisateur\\Documents\\Projet_MLOps\\Champy_Classifier\\data\\" # Chemin vers les datasets csv + images raw => processed => split

In [4]:
# Paramètres


# Ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Seed pour reproductibilité
SEED = 42

# Extensions autorisées
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

# False = refuse d'écraser si output_root existe déjà
# True  = supprime puis recrée output_root
OVERWRITE_OUTPUT = False

# Fichiers générés
MANIFEST_FILENAME = "split_manifest.csv"
SUMMARY_FILENAME = "split_summary.csv"

In [5]:
# Définitions des fonctions :

dup_suffix_re = re.compile(r"^(?P<base>.+?)(?:_(?P<n>\d+))?$")


def base_id_only(p: Path) -> str:
    """
    Exemples :
    705.jpg    -> 705
    705_1.jpg  -> 705
    705_12.jpg -> 705
    """
    stem = p.stem
    m = dup_suffix_re.match(stem)
    return m.group("base") if m else stem


def group_id_from_path(p: Path) -> str:
    """
    Clé de groupe = classe + base_id
    Ex: bolet_bai::705
    """
    return f"{p.parent.name}::{base_id_only(p)}"


def list_all_images(rep_img_path: Path):
    """
    Liste uniquement les images situées directement dans :
    rep_img/<classe>/<fichier_image>

    Ignore les niveaux plus profonds.
    """
    return [
        p for p in rep_img_path.rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTS
        and p.parent.parent == rep_img_path
    ]


def build_groups(rep_img_path: Path):
    """
    Construit :
    - photos_totales
    - class_names
    - class_to_idx
    - groups : dict[group_id] -> list[Path]
    - class_to_group_ids : dict[class_name] -> list[group_id]
    """
    photos_totales = list_all_images(rep_img_path)

    if not photos_totales:
        raise ValueError(f"Aucune image trouvée dans {rep_img_path}")

    class_names = sorted({p.parent.name for p in photos_totales})
    class_to_idx = {c: i for i, c in enumerate(class_names)}

    groups = {}
    for p in photos_totales:
        cls = p.parent.name
        gid = group_id_from_path(p)
        groups.setdefault(gid, []).append(p)

    # Tri interne pour reproductibilité
    for gid in groups:
        groups[gid] = sorted(groups[gid], key=lambda x: x.name.lower())

    class_to_group_ids = {cls: [] for cls in class_names}
    for gid in sorted(groups.keys()):
        cls = gid.split("::", 1)[0]
        class_to_group_ids[cls].append(gid)

    return photos_totales, class_names, class_to_idx, groups, class_to_group_ids


def stratified_split_groups_by_class(class_to_group_ids, seed=42):
    """
    Split stratifié par classe :
    pour chaque classe, on mélange ses groupes puis on applique les ratios.

    Retourne :
    - train_g, val_g, test_g : ensembles globaux de group_ids
    - per_class_stats : liste de stats par classe
    """
    train_g, val_g, test_g = set(), set(), set()
    per_class_stats = []

    for class_idx, cls in enumerate(sorted(class_to_group_ids.keys())):
        cls_group_ids = np.array(sorted(class_to_group_ids[cls]), dtype=object)

        rng = np.random.default_rng(seed + class_idx)
        rng.shuffle(cls_group_ids)

        n = len(cls_group_ids)
        n_train = int(TRAIN_RATIO * n)
        n_val = int(VAL_RATIO * n)

        cls_train = set(cls_group_ids[:n_train])
        cls_val = set(cls_group_ids[n_train:n_train + n_val])
        cls_test = set(cls_group_ids[n_train + n_val:])

        # Tout ce qui reste va au test, ce qui garantit la somme
        assert len(cls_train | cls_val | cls_test) == n
        assert cls_train.isdisjoint(cls_val)
        assert cls_train.isdisjoint(cls_test)
        assert cls_val.isdisjoint(cls_test)

        train_g.update(cls_train)
        val_g.update(cls_val)
        test_g.update(cls_test)

        per_class_stats.append({
            "class_name": cls,
            "nb_groups_total": n,
            "nb_groups_train": len(cls_train),
            "nb_groups_val": len(cls_val),
            "nb_groups_test": len(cls_test),
        })

    assert train_g.isdisjoint(val_g)
    assert train_g.isdisjoint(test_g)
    assert val_g.isdisjoint(test_g)

    return train_g, val_g, test_g, per_class_stats


def prepare_output_dir(output_root_path: Path, overwrite: bool = False):
    if output_root_path.exists():
        if overwrite:
            shutil.rmtree(output_root_path)
        else:
            raise FileExistsError(
                f"Le dossier de sortie existe déjà : {output_root_path}\n"
                f"Passe OVERWRITE_OUTPUT=True pour l'écraser."
            )

    (output_root_path / "train").mkdir(parents=True, exist_ok=True)
    (output_root_path / "val").mkdir(parents=True, exist_ok=True)
    (output_root_path / "test").mkdir(parents=True, exist_ok=True)


def copy_group_files_and_build_manifest(
    group_ids,
    groups,
    split_name: str,
    output_root_path: Path,
    class_to_idx: dict,
    rep_img_path: Path,
):
    manifest_rows = []
    n_files = 0
    n_groups = 0

    for gid in sorted(group_ids):
        files = groups[gid]
        cls = gid.split("::", 1)[0]
        class_idx = class_to_idx[cls]

        class_dir = output_root_path / split_name / cls
        class_dir.mkdir(parents=True, exist_ok=True)

        for src in files:
            dst = class_dir / src.name
            shutil.copy2(src, dst)

            manifest_rows.append({
                "split": split_name,
                "class_name": cls,
                "class_idx": class_idx,
                "group_id": gid,
                "base_id": base_id_only(src),
                "filename": src.name,
                "extension": src.suffix.lower(),
                "src_path": src.resolve().as_posix(),
                "dst_path": dst.resolve().as_posix(),
                "src_rel_to_root": src.relative_to(rep_img_path).as_posix(),
                "dst_rel_to_output": dst.relative_to(output_root_path).as_posix(),
            })
            n_files += 1

        n_groups += 1

    return manifest_rows, n_groups, n_files


def write_manifest_csv(rows, csv_path: Path):
    if not rows:
        raise ValueError("Aucune ligne à écrire dans le manifest CSV.")

    fieldnames = [
        "split",
        "class_name",
        "class_idx",
        "group_id",
        "base_id",
        "filename",
        "extension",
        "src_path",
        "dst_path",
        "src_rel_to_root",
        "dst_rel_to_output",
    ]

    rows_sorted = sorted(
        rows,
        key=lambda r: (
            r["split"],
            r["class_name"],
            r["group_id"],
            r["filename"],
        )
    )

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows_sorted)


def build_summary_rows(manifest_rows):
    agg = {}

    for row in manifest_rows:
        key = (row["split"], row["class_name"], row["class_idx"])
        if key not in agg:
            agg[key] = {
                "split": row["split"],
                "class_name": row["class_name"],
                "class_idx": row["class_idx"],
                "group_ids": set(),
                "nb_images": 0,
            }

        agg[key]["group_ids"].add(row["group_id"])
        agg[key]["nb_images"] += 1

    summary_rows = []
    for _, v in agg.items():
        summary_rows.append({
            "split": v["split"],
            "class_name": v["class_name"],
            "class_idx": v["class_idx"],
            "nb_groups": len(v["group_ids"]),
            "nb_images": v["nb_images"],
        })

    summary_rows = sorted(summary_rows, key=lambda r: (r["split"], r["class_name"]))
    return summary_rows


def write_summary_csv(summary_rows, csv_path: Path):
    fieldnames = ["split", "class_name", "class_idx", "nb_groups", "nb_images"]

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(summary_rows)


def print_class_split_stats(per_class_stats):
    print("\n=== Répartition des groupes par classe ===")
    print(f"{'Classe':30} {'Total':>7} {'Train':>7} {'Val':>7} {'Test':>7}")
    print("-" * 65)
    for row in per_class_stats:
        print(
            f"{row['class_name'][:30]:30} "
            f"{row['nb_groups_total']:>7} "
            f"{row['nb_groups_train']:>7} "
            f"{row['nb_groups_val']:>7} "
            f"{row['nb_groups_test']:>7}"
        )


def print_global_summary(manifest_rows):
    stats = {}
    for row in manifest_rows:
        split = row["split"]
        stats.setdefault(split, {"images": 0, "groups": set(), "classes": set()})
        stats[split]["images"] += 1
        stats[split]["groups"].add(row["group_id"])
        stats[split]["classes"].add(row["class_name"])

    print("\n=== Résumé global à partir du manifest ===")
    for split in ["train", "val", "test"]:
        if split in stats:
            print(
                f"{split:<5} -> "
                f"{len(stats[split]['groups']):>5} groupes | "
                f"{stats[split]['images']:>6} images | "
                f"{len(stats[split]['classes']):>3} classes"
            )

In [6]:
# Script principal :

def main():
    total_ratio = TRAIN_RATIO + VAL_RATIO + TEST_RATIO
    if not np.isclose(total_ratio, 1.0):
        raise ValueError(f"La somme des ratios doit valoir 1.0, ici = {total_ratio}")

    rep_img_path = Path(rep_data + "processed\\")
    output_root_path = Path(rep_data + "split\\")

    if not rep_img_path.exists():
        raise FileNotFoundError(f"Dossier source introuvable : {rep_img_path}")

    print("=== Analyse du dossier source ===")
    (
        photos_totales,
        class_names,
        class_to_idx,
        groups,
        class_to_group_ids,
    ) = build_groups(rep_img_path)

    print(f"Nb images trouvées : {len(photos_totales)}")
    print(f"Nb classes         : {len(class_names)}")
    print(f"Nb groupes         : {len(groups)}")
    print("\nExemples de classes :")
    print(class_names[:10])

    train_g, val_g, test_g, per_class_stats = stratified_split_groups_by_class(
        class_to_group_ids,
        seed=SEED,
    )

    print_class_split_stats(per_class_stats)

    print("\n=== Vérification des splits globaux ===")
    print("train ∩ val  =", train_g.isdisjoint(val_g))
    print("train ∩ test =", train_g.isdisjoint(test_g))
    print("val   ∩ test =", val_g.isdisjoint(test_g))
    print("Total groupes répartis =", len(train_g | val_g | test_g))

    print("\n=== Préparation du dossier de sortie ===")
    prepare_output_dir(output_root_path, overwrite=OVERWRITE_OUTPUT)

    print("\n=== Copie des fichiers et construction du manifest ===")
    all_rows = []

    rows_train, train_groups, train_files = copy_group_files_and_build_manifest(
        train_g, groups, "train", output_root_path, class_to_idx, rep_img_path
    )
    rows_val, val_groups, val_files = copy_group_files_and_build_manifest(
        val_g, groups, "val", output_root_path, class_to_idx, rep_img_path
    )
    rows_test, test_groups, test_files = copy_group_files_and_build_manifest(
        test_g, groups, "test", output_root_path, class_to_idx, rep_img_path
    )

    all_rows.extend(rows_train)
    all_rows.extend(rows_val)
    all_rows.extend(rows_test)

    print(f"Train : {train_groups} groupes copiés, {train_files} images")
    print(f"Val   : {val_groups} groupes copiés, {val_files} images")
    print(f"Test  : {test_groups} groupes copiés, {test_files} images")

    total_copied = train_files + val_files + test_files
    print(f"\nTotal images copiées : {total_copied}")

    if total_copied != len(photos_totales):
        print("⚠ Attention : le nombre d'images copiées diffère du nombre d'images source.")
    else:
        print("✅ Toutes les images ont bien été réparties.")

    manifest_path = output_root_path / MANIFEST_FILENAME
    write_manifest_csv(all_rows, manifest_path)
    print(f"\n✅ Manifest CSV écrit : {manifest_path}")

    summary_rows = build_summary_rows(all_rows)
    summary_path = output_root_path / SUMMARY_FILENAME
    write_summary_csv(summary_rows, summary_path)
    print(f"✅ Summary CSV écrit  : {summary_path}")

    print_global_summary(all_rows)

    print("\n✅ Terminé.")
    print(f"Données splittées dans : {output_root_path}")



In [8]:
#if __name__ == "__main__":
#    main()

main()

=== Analyse du dossier source ===
Nb images trouvées : 25850
Nb classes         : 30
Nb groupes         : 12160

Exemples de classes :
['Agaricus bisporus', 'Agaricus campestris', 'Amanita muscaria', 'Amanita phalloides', 'Amanita rubescens', 'Armillaria mellea', 'Auricularia auricula-judae', 'Boletus edulis', 'Cantharellus cibarius', 'Clitocybe nebularis']

=== Répartition des groupes par classe ===
Classe                           Total   Train     Val    Test
-----------------------------------------------------------------
Agaricus bisporus                   82      57      12      13
Agaricus campestris                430     301      64      65
Amanita muscaria                   900     630     135     135
Amanita phalloides                 900     630     135     135
Amanita rubescens                  482     337      72      73
Armillaria mellea                  495     346      74      75
Auricularia auricula-judae          91      63      13      15
Boletus edulis            